# Training, optimizer comparison and momentum

EN3150 Assignment 03 - Xception. Tasks 3 and 4.

The training loop saves the lowest-validation-loss checkpoint. Source also includes the controlled Adam/SGD/SGD+momentum and momentum-ablation drivers. The default workflow only runs a two-epoch offline smoke check; opt in to full experiments after notebook 01 prepares the images. Existing results under results/ are the report's recorded runs, not new notebook execution outputs.

## Running this notebook

Use Python 3.10+ with the project dependencies. Extract the whole ZIP and open this notebook beside raw_code/. On Colab, upload and extract the complete ZIP before running. Set INSTALL_DEPENDENCIES = True in the setup cell for a fresh runtime. Run cells in order with a fresh kernel for each notebook.

The writefile cells below contain the original Python source, including comments. They save that code into the bundled package so relative imports and CLI entry points behave exactly as in the repository. The complete package is already included: the notebooks can be run independently.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Extract the entire ZIP first. This also works from notebooks/submission in Git.
start = Path.cwd().resolve()
candidates = [start, *start.parents]
PROJECT = next((base for parent in candidates
                for base in (parent / "raw_code", parent)
                if (base / "src/pr_edge").is_dir()
                and (base / "configs/base.yaml").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Extract the complete code ZIP, including raw_code, first.")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
sys.path.insert(0, str(PROJECT))

# Enable once in a fresh Python 3.10+ / Colab environment (requires internet).
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

def run_script(script, *args):
    subprocess.run([sys.executable, str(PROJECT / script), *map(str, args)],
                   cwd=PROJECT, check=True)

print("Project:", PROJECT)


## Source: `src/pr_edge/train.py`


In [ ]:
%%writefile src/pr_edge/train.py
"""Training loop.

One loop serves every experiment in the project: both custom models, the three
optimizer variants, and the fine-tuned SOTA models. Anything model-specific
belongs in ``models/``, not here.

Model selection (#6): the checkpoint kept is the epoch with the lowest
validation loss, not the last one, and those weights are restored into the
model before ``train`` returns. The caller evaluates and profiles the in-memory
model, so without the restore every test metric would describe the final epoch
while ``model.pt`` held a different one. The chosen epoch is recorded in
``selection.json`` so the report can cite it.
"""

from __future__ import annotations

import json
import math
import os
import time
from collections.abc import Callable
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from .config import Config, OptimizerConfig


def build_optimizer(model: nn.Module, cfg: OptimizerConfig) -> torch.optim.Optimizer:
    """Construct the optimizer named in the config.

    Only Adam and SGD, because task 3 compares exactly those: the chosen
    optimizer against plain SGD and SGD with momentum. ``momentum`` is ignored
    by Adam, which is why the sweep in #13 must vary only the optimizer block
    and nothing else.
    """
    params = [p for p in model.parameters() if p.requires_grad]
    name = cfg.name.lower()

    if name == "adam":
        return torch.optim.Adam(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    if name == "sgd":
        return torch.optim.SGD(
            params, lr=cfg.lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay
        )
    raise ValueError(f"unknown optimizer '{cfg.name}'. Known: adam, sgd")


def _run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    optimizer: torch.optim.Optimizer | None = None,
) -> tuple[float, float]:
    """One pass over a loader. Trains when an optimizer is given, else evaluates."""
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    correct = 0
    seen = 0

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, labels)

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            batch = labels.size(0)
            total_loss += loss.item() * batch
            correct += (logits.argmax(dim=1) == labels).sum().item()
            seen += batch

    return total_loss / seen, correct / seen


# The selection criterion, recorded in selection.json. Validation LOSS rather
# than accuracy: accuracy only changes when an argmax flips, so it plateaus and
# ties across epochs while the network is already growing confidently wrong on
# held-out images. Loss is continuous and shows that overfitting first.
SELECTION_CRITERION = "val_loss"


def _atomic_write(path: Path, write: Callable[[Path], None]) -> None:
    """Write via a sibling temp file, then rename over ``path``.

    A Colab runtime can disconnect at any moment. ``os.replace`` is atomic, so
    ``path`` always holds either the previous complete file or the new one -
    never a truncated checkpoint that fails to load at evaluation time.
    """
    temporary = path.with_name(f".{path.name}.tmp")
    try:
        write(temporary)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def _write_json(path: Path, data: object) -> None:
    _atomic_write(path, lambda tmp: tmp.write_text(json.dumps(data, indent=2), encoding="utf-8"))


def train(
    model: nn.Module,
    cfg: Config,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    out_dir: Path,
) -> list[dict]:
    """Train for ``cfg.training.epochs`` and write ``history.json``.

    Returns the history records. ``epoch_seconds`` times the training pass only
    and deliberately excludes validation, so the "training time per epoch"
    column in the report means what it says.

    On return ``model`` holds the weights of the epoch with the lowest
    validation loss (ties go to the earlier epoch), which is also what
    ``model.pt`` contains. ``history.json`` is rewritten every epoch and
    ``model.pt`` on every improvement, so an interrupted run still leaves its
    best checkpoint and a readable history behind.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = build_optimizer(model, cfg.optimizer)
    model.to(device)

    checkpoint = out_dir / "model.pt"
    selection_file = out_dir / "selection.json"
    # A rerun into the same directory must not leave a previous run's weights
    # or selection beside this run's history if it fails before its first save.
    checkpoint.unlink(missing_ok=True)
    selection_file.unlink(missing_ok=True)

    history: list[dict] = []
    best: dict | None = None

    for epoch in range(1, cfg.training.epochs + 1):
        start = time.perf_counter()
        train_loss, train_acc = _run_epoch(model, train_loader, criterion, device, optimizer)
        epoch_seconds = time.perf_counter() - start

        val_loss, val_acc = _run_epoch(model, val_loader, criterion, device)

        record = {
            "epoch": epoch,
            "train_loss": round(train_loss, 6),
            "train_acc": round(train_acc, 6),
            "val_loss": round(val_loss, 6),
            "val_acc": round(val_acc, 6),
            "epoch_seconds": round(epoch_seconds, 4),
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(record)

        # Compare the value as recorded in history.json, so anyone can re-derive
        # the choice from that file alone. Strict "<" keeps the earlier epoch on
        # a tie. A diverged (NaN/inf) epoch is never selected, but the run keeps
        # going: divergence is itself a result the optimizer study reports.
        improved = math.isfinite(record["val_loss"]) and (
            best is None or record["val_loss"] < best["val_loss"]
        )
        if improved:
            best = record
            _atomic_write(checkpoint, lambda tmp: torch.save(model.state_dict(), tmp))
        _write_json(out_dir / "history.json", history)

        print(
            f"epoch {epoch:3d}/{cfg.training.epochs}  "
            f"train {train_loss:.4f}/{train_acc:.3f}  "
            f"val {val_loss:.4f}/{val_acc:.3f}  "
            f"{epoch_seconds:.1f}s" + ("  *best" if improved else "")
        )

    if best is None:
        raise RuntimeError(
            "no epoch produced a finite validation loss - the run diverged from the "
            "start, so there is no checkpoint to evaluate. history.json is kept."
        )

    # Restore from the file rather than an in-memory copy, so the weights that
    # get evaluated and profiled are provably the ones saved in model.pt.
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))

    _write_json(selection_file, {
        "criterion": SELECTION_CRITERION,
        "best_epoch": best["epoch"],
        "best_val_loss": best["val_loss"],
        "best_val_acc": best["val_acc"],
        "epochs_trained": len(history),
    })
    print(f"[select] epoch {best['epoch']}/{len(history)} by lowest val_loss "
          f"({best['val_loss']:.4f}, val acc {best['val_acc']:.3f})")

    return history


## Source: `scripts/run_experiment.py`


In [ ]:
%%writefile scripts/run_experiment.py
#!/usr/bin/env python
"""Run one experiment end to end and write a complete run directory.

    python scripts/run_experiment.py --config configs/model_b.yaml
    python scripts/run_experiment.py --config configs/model_b.yaml --smoke

CLI flags override the config file. The RESOLVED values - after overrides -
are what get dumped to ``config.yaml`` in the run directory, so a run is always
reproducible from its own output.
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from pr_edge.config import dump_config, load_config  # noqa: E402
from pr_edge.data import get_dataloaders, get_synthetic_dataloaders  # noqa: E402
from pr_edge.device import get_device, set_seed  # noqa: E402
from pr_edge.evaluate import evaluate  # noqa: E402
from pr_edge.models import build_model  # noqa: E402
from pr_edge.paths import run_dir  # noqa: E402
from pr_edge.profile import profile_model  # noqa: E402
from pr_edge.train import train  # noqa: E402

# Smoke mode overrides, frozen in docs/interfaces.md section 4. These are not
# tunable: CI relies on the run finishing in under 60 seconds on a CPU runner.
SMOKE_OVERRIDES = {"epochs": 2, "batch_size": 16, "num_workers": 0, "device": "cpu", "smoke": True}


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--config", required=True, type=Path, help="path to a YAML config in configs/")
    parser.add_argument("--smoke", action="store_true",
                        help="synthetic data, 2 epochs, CPU; writes results/_smoke_<run_id>/")
    parser.add_argument("--epochs", type=int, default=None, help="override training.epochs")
    parser.add_argument("--device", type=str, default=None, help="override device (auto|cpu|cuda|mps)")
    parser.add_argument("--run-id", type=str, default=None, help="override the output directory name")
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)

    overrides: dict = {"epochs": args.epochs, "device": args.device}
    if args.smoke:
        overrides.update(SMOKE_OVERRIDES)

    cfg = load_config(args.config, overrides)
    run_id = args.run_id or cfg.run_id
    out_dir = run_dir(run_id, smoke=cfg.training.smoke)

    set_seed(cfg.seed)
    device = get_device(cfg.device)
    print(f"[run] {run_id}  device={device}  epochs={cfg.training.epochs}  seed={cfg.seed}")

    # The resolved config is written FIRST, so a crashed run still leaves
    # behind enough to reproduce what was attempted.
    dump_config(cfg, out_dir / "config.yaml")

    loaders = get_synthetic_dataloaders(cfg) if cfg.training.smoke else get_dataloaders(cfg)
    train_loader, val_loader, test_loader = loaders

    model = build_model(cfg)

    history = train(model, cfg, train_loader, val_loader, device, out_dir)

    metrics = evaluate(model, test_loader, cfg, device, out_dir)
    print(f"[test] accuracy={metrics['accuracy']:.4f}  "
          f"macro_precision={metrics['macro_precision']:.4f}  "
          f"macro_recall={metrics['macro_recall']:.4f}")

    mean_epoch = sum(r["epoch_seconds"] for r in history) / len(history)
    profile = profile_model(model, cfg, mean_epoch, device, out_dir)
    print(f"[profile] params={profile['trainable_params']:,}  "
          f"size={profile['disk_size_kb']:.1f} KB  "
          f"macs={profile['macs']:,}  "
          f"{profile['mean_epoch_seconds']:.2f}s/epoch")

    print(f"[done] {out_dir}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Source: `scripts/run_optimizer_sweep.py`


In [ ]:
%%writefile scripts/run_optimizer_sweep.py
#!/usr/bin/env python
"""Run the Task 3 optimizer comparison and plot validation loss.

The three experiments intentionally share the same model, seed, data order,
batch size, epoch budget, and device.  Only the optimizer block differs:

    python scripts/run_optimizer_sweep.py --device cuda
    python scripts/run_optimizer_sweep.py --smoke

Full runs write ``report/figures/optimizer_comparison.png``.  Smoke runs keep
their meaningless random-data figure under the gitignored smoke-results tree.
"""

from __future__ import annotations

import argparse
import json
import sys
from dataclasses import asdict
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pr_edge.config import Config, load_config  # noqa: E402
from pr_edge.paths import CONFIGS_DIR, REPORT_DIR, run_dir  # noqa: E402
from scripts import run_experiment  # noqa: E402

SWEEP_CONFIGS = (
    CONFIGS_DIR / "model_b.yaml",
    CONFIGS_DIR / "model_b_sgd.yaml",
    CONFIGS_DIR / "model_b_sgdm.yaml",
)

LABELS = {
    "model_b": "Adam (lr=0.001)",
    "model_b_sgd": "SGD (lr=0.01)",
    "model_b_sgdm": "SGD + momentum 0.9 (lr=0.01)",
}


def _validate_sweep_configs(configs: list[Config]) -> None:
    """Fail before training if anything except run ID and optimizer differs."""
    if [cfg.run_id for cfg in configs] != list(LABELS):
        raise ValueError(f"optimizer sweep requires run IDs {list(LABELS)}")

    common = []
    for cfg in configs:
        values = asdict(cfg)
        values.pop("run_id")
        values.pop("optimizer")
        common.append(values)
    if any(values != common[0] for values in common[1:]):
        raise ValueError(
            "optimizer sweep configs differ outside run_id and optimizer; "
            "the comparison would not isolate optimizer choice"
        )

    observed = [
        (cfg.optimizer.name, cfg.optimizer.lr, cfg.optimizer.momentum,
         cfg.optimizer.weight_decay)
        for cfg in configs
    ]
    expected = [
        ("adam", 0.001, 0.0, 0.0),
        ("sgd", 0.01, 0.0, 0.0),
        ("sgd", 0.01, 0.9, 0.0),
    ]
    if observed != expected:
        raise ValueError(f"unexpected optimizer sweep blocks: {observed!r}")


def _plot_validation_losses(histories: dict[str, Path], out: Path) -> Path:
    """Plot one validation-loss curve per optimizer from committed run data."""
    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    for run_id, history_path in histories.items():
        records = json.loads(history_path.read_text(encoding="utf-8"))
        if not records:
            raise ValueError(f"empty history: {history_path}")
        ax.plot(
            [record["epoch"] for record in records],
            [record["val_loss"] for record in records],
            marker="o",
            markersize=3,
            linewidth=1.8,
            label=LABELS[run_id],
        )

    ax.set_title("Model B optimizer comparison")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation loss")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return out


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--smoke",
        action="store_true",
        help="run all three variants on synthetic data for two CPU epochs",
    )
    parser.add_argument("--epochs", type=int, default=None, help="override training.epochs")
    parser.add_argument(
        "--device", type=str, default=None, help="override device (auto|cpu|cuda|mps)"
    )
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    configs = [load_config(path) for path in SWEEP_CONFIGS]
    _validate_sweep_configs(configs)

    histories: dict[str, Path] = {}
    for path, cfg in zip(SWEEP_CONFIGS, configs, strict=True):
        runner_args = ["--config", str(path)]
        if args.smoke:
            runner_args.append("--smoke")
        if args.epochs is not None:
            runner_args.extend(["--epochs", str(args.epochs)])
        if args.device is not None:
            runner_args.extend(["--device", args.device])

        print(f"\n[sweep] starting {LABELS[cfg.run_id]}")
        exit_code = run_experiment.main(runner_args)
        if exit_code != 0:
            return exit_code
        histories[cfg.run_id] = run_dir(cfg.run_id, smoke=args.smoke) / "history.json"

    if args.smoke:
        figure = run_dir("optimizer_sweep", smoke=True) / "optimizer_comparison.png"
    else:
        figure = REPORT_DIR / "figures" / "optimizer_comparison.png"
    _plot_validation_losses(histories, figure)
    print(f"[sweep] figure={figure}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Source: `scripts/run_momentum_ablation.py`


In [ ]:
%%writefile scripts/run_momentum_ablation.py
#!/usr/bin/env python
"""Run the Task 3 momentum ablation and plot validation loss.

All four experiments use Model B with SGD at learning rate 0.01.  Momentum is
the only methodological variable:

    python scripts/run_momentum_ablation.py --device cuda --reuse-existing
    python scripts/run_momentum_ablation.py --smoke

The reusable 0.0 and 0.9 runs are produced by issue #13.  Reuse is deliberately
opt-in and validates the complete run plus its resolved configuration before a
history is accepted.  Full runs write ``report/figures/momentum_ablation.png``.
"""

from __future__ import annotations

import argparse
import json
import sys
import tempfile
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import yaml  # noqa: E402

PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pr_edge.config import Config, dump_config, load_config  # noqa: E402
from pr_edge.paths import CONFIGS_DIR, REPORT_DIR, RESULTS_DIR, run_dir  # noqa: E402
from pr_edge.schemas import validate_run_dir  # noqa: E402
from scripts import run_experiment  # noqa: E402

BASE_CONFIG = CONFIGS_DIR / "model_b_sgd.yaml"
MOMENTUM_RUNS = {
    0.0: "momentum_m000",
    0.5: "momentum_m050",
    0.9: "momentum_m090",
    0.99: "momentum_m099",
}
REUSE_RUNS = {0.0: "model_b_sgd", 0.9: "model_b_sgdm"}
LABELS = {momentum: f"m={momentum:g}" for momentum in MOMENTUM_RUNS}


def _build_configs() -> list[Config]:
    """Derive the four configs without adding non-canonical YAML files."""
    base = load_config(BASE_CONFIG)
    return [
        replace(
            base,
            run_id=run_id,
            optimizer=replace(base.optimizer, momentum=momentum),
        )
        for momentum, run_id in MOMENTUM_RUNS.items()
    ]


def _validate_ablation_configs(configs: list[Config]) -> None:
    """Fail unless run ID and momentum are the only differences."""
    if [cfg.run_id for cfg in configs] != list(MOMENTUM_RUNS.values()):
        raise ValueError("momentum ablation has unexpected run IDs")

    common = []
    for cfg in configs:
        values = asdict(cfg)
        values.pop("run_id")
        values["optimizer"].pop("momentum")
        common.append(values)
    if any(values != common[0] for values in common[1:]):
        raise ValueError(
            "momentum ablation configs differ outside run_id and optimizer.momentum"
        )

    observed = [
        (cfg.optimizer.name, cfg.optimizer.lr, cfg.optimizer.momentum, cfg.optimizer.weight_decay)
        for cfg in configs
    ]
    expected = [("sgd", 0.01, momentum, 0.0) for momentum in MOMENTUM_RUNS]
    if observed != expected:
        raise ValueError(f"unexpected momentum ablation optimizer blocks: {observed!r}")


def _resolved_config(cfg: Config, args: argparse.Namespace) -> dict:
    """Expected config.yaml contents after CLI overrides."""
    expected = asdict(cfg)
    if args.device is not None:
        expected["device"] = args.device
    if args.epochs is not None:
        expected["training"]["epochs"] = args.epochs
    if args.smoke:
        expected["device"] = "cpu"
        expected["data"]["batch_size"] = 16
        expected["data"]["num_workers"] = 0
        expected["training"]["epochs"] = 2
        expected["training"]["smoke"] = True
    return expected


def _reusable_history(momentum: float, cfg: Config, args: argparse.Namespace) -> Path | None:
    """Return a validated existing history, or ``None`` when no run exists."""
    reuse_id = REUSE_RUNS.get(momentum, cfg.run_id)
    directory = RESULTS_DIR / (f"_smoke_{reuse_id}" if args.smoke else reuse_id)
    if not directory.exists():
        return None
    errors = validate_run_dir(directory)
    if errors:
        raise ValueError(f"cannot reuse incomplete run {reuse_id}: {errors}")
    actual = yaml.safe_load((directory / "config.yaml").read_text(encoding="utf-8"))
    expected = _resolved_config(cfg, args)
    expected["run_id"] = reuse_id
    if actual != expected:
        raise ValueError(f"cannot reuse {reuse_id}: resolved config does not match ablation")
    return directory / "history.json"


def _plot_validation_losses(histories: dict[float, Path], out: Path) -> Path:
    """Overlay one validation-loss curve per momentum value."""
    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    for momentum, history_path in histories.items():
        records = json.loads(history_path.read_text(encoding="utf-8"))
        if not records:
            raise ValueError(f"empty history: {history_path}")
        ax.plot(
            [record["epoch"] for record in records],
            [record["val_loss"] for record in records],
            marker="o",
            markersize=3,
            linewidth=1.8,
            label=LABELS[momentum],
        )

    ax.set_title("Model B momentum ablation (SGD, lr=0.01)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation loss")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return out


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--smoke", action="store_true", help="two CPU epochs on synthetic data")
    parser.add_argument("--reuse-existing", action="store_true", help="reuse only schema/config-valid runs")
    parser.add_argument("--epochs", type=int, default=None, help="override training.epochs")
    parser.add_argument("--device", type=str, default=None, help="override device (auto|cpu|cuda|mps)")
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    configs = _build_configs()
    _validate_ablation_configs(configs)

    histories: dict[float, Path] = {}
    with tempfile.TemporaryDirectory(prefix="momentum-configs-") as temp:
        temp_dir = Path(temp)
        for (momentum, _), cfg in zip(MOMENTUM_RUNS.items(), configs, strict=True):
            if args.reuse_existing:
                existing = _reusable_history(momentum, cfg, args)
                if existing is not None:
                    histories[momentum] = existing
                    print(f"[momentum] reusing {existing.parent.name} ({LABELS[momentum]})")
                    continue

            config_path = temp_dir / f"{cfg.run_id}.yaml"
            dump_config(cfg, config_path)
            runner_args = ["--config", str(config_path)]
            if args.smoke:
                runner_args.append("--smoke")
            if args.epochs is not None:
                runner_args.extend(["--epochs", str(args.epochs)])
            if args.device is not None:
                runner_args.extend(["--device", args.device])

            print(f"\n[momentum] starting {LABELS[momentum]}")
            exit_code = run_experiment.main(runner_args)
            if exit_code != 0:
                return exit_code
            histories[momentum] = run_dir(cfg.run_id, smoke=args.smoke) / "history.json"

    if args.smoke:
        figure = run_dir("momentum_ablation", smoke=True) / "momentum_ablation.png"
    else:
        figure = REPORT_DIR / "figures" / "momentum_ablation.png"
    _plot_validation_losses(histories, figure)
    print(f"[momentum] figure={figure}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Run / inspect results


In [ ]:
# Offline integration check: synthetic data and a tiny stand-in network.
# These smoke results are NOT the reported architecture accuracies.
run_script("scripts/run_experiment.py", "--config", "configs/model_b.yaml", "--smoke")

# Real runs require prepared EuroSAT and may take hours. Prefer a GPU runtime.
# Prefix outputs so the supplied benchmark records remain intact.
RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    for name in ("model_a", "model_b", "model_b_sgd", "model_b_sgdm",
                 "model_a_100", "model_b_100"):
        run_script("scripts/run_experiment.py", "--config", f"configs/{name}.yaml",
                   "--device", "auto", "--run-id", f"rerun_{name}")
